In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/nifty100.db")

pl = pd.read_sql(
    "SELECT * FROM profitandloss",
    conn
)

bs = pd.read_sql(
    "SELECT * FROM balancesheet",
    conn
)

In [3]:
import sys
from pathlib import Path

ROOT = Path("../").resolve()
sys.path.append(str(ROOT))

from src.analytics.ratios import (
    calculate_npm,
    calculate_opm,
    calculate_roe,
    calculate_roce,
    validate_opm
)

In [4]:
validation = []

for _, row in pl.iterrows():

    computed, diff, valid = validate_opm(
        row["operating_profit"],
        row["sales"],
        row["opm_percentage"]
    )

    validation.append({
        "company_id": row["company_id"],
        "year": row["year"],
        "source_opm": row["opm_percentage"],
        "computed_opm": computed,
        "difference": diff,
        "valid": valid
    })

validation_df = pd.DataFrame(validation)

validation_df.head()

,company_id,year,source_opm,computed_opm,difference,valid
0,ABB,2012-12,12.0,12.220206,0.220206,True
1,ABB,2014-03,12.0,11.731107,0.268893,True
2,ABB,2015-03,14.0,13.630406,0.369594,True
3,ABB,2016-03,14.0,13.963275,0.036725,True
4,ABB,2017-03,14.0,13.709955,0.290045,True


In [5]:
print(validation_df["valid"].value_counts())

valid
True     915
False    234
Name: count, dtype: int64


In [6]:
validation_df[
    validation_df["valid"] == False
]

,company_id,year,source_opm,computed_opm,difference,valid
23,ADANIENSOL,2024-03,30.0,34.389113,4.389113,False
133,AXISBANK,2013-03,1353.0,30.581614,1322.418386,False
134,AXISBANK,2014-03,2307.0,31.474169,2275.525831,False
135,AXISBANK,2015-03,3097.0,31.362214,3065.637786,False
136,AXISBANK,2016-03,3466.0,32.611984,3433.388016,False
...,...,...,...,...,...,...
1159,INDIGO,2021-03,2.0,99.979510,97.979510,False
1160,INDIGO,2022-03,558.0,97.848135,460.151865,False
1161,INDIGO,2023-03,6521.0,88.024832,6432.975168,False
1162,INDIGO,2024-03,16331.0,76.298909,16254.701091,False


In [7]:
pl[
    pl["company_id"] == "AXISBANK"
][[
    "year",
    "sales",
    "operating_profit",
    "opm_percentage"
]]

,year,sales,operating_profit,opm_percentage
133,2013-03,27183.0,8313.0,1353.0
134,2014-03,30641.0,9644.0,2307.0
135,2015-03,35479.0,11127.0,3097.0
136,2016-03,40988.0,13367.0,3466.0
137,2017-03,44542.0,23808.0,-5715.0
138,2018-03,45780.0,28895.0,-10277.0
139,2019-03,54986.0,27155.0,-5447.0
140,2020-03,62635.0,35066.0,-9859.0
141,2021-03,63346.0,31749.0,-2510.0
142,2022-03,67377.0,29962.0,3170.0


In [8]:
pl[
    pl["company_id"] == "INDIGO"
][[
    "year",
    "sales",
    "operating_profit",
    "opm_percentage"
]]

,year,sales,operating_profit,opm_percentage
1151,2013-03,9203.0,8348.0,855.0
1152,2014-03,11117.0,10684.0,433.0
1153,2015-03,13925.0,12022.0,1904.0
1154,2016-03,16140.0,12976.0,3164.0
1155,2017-03,18580.0,16362.0,2219.0
1156,2018-03,23021.0,19991.0,3030.0
1157,2019-03,28497.0,28648.0,-151.0
1158,2020-03,35756.0,31687.0,4069.0
1159,2021-03,14641.0,14638.0,2.0
1160,2022-03,25931.0,25373.0,558.0


1. The calculate_opm() implementation is correct.
2. The cross-validation is working exactly as intended.
3. The source opm_percentage values are inconsistent for some companies.

In [9]:
validation_df["valid"].value_counts(dropna=False)

valid
True     915
False    234
None      15
Name: count, dtype: int64

In [12]:
validation_df[
    validation_df["difference"] > 1
].to_csv(
    "../output/opm_crosscheck.csv",
    index=False
)

from pathlib import Path

Path("../output").mkdir(
    exist_ok=True
)